# Интерактивное исследование датасета в Google Colab

Загрузите созданный файл `ai_phonetics_data_v1.zip` через окно Colab либо положите его в Google Drive. Notebook распакует набор, восстановит синтетические изображения по seed и позволит смотреть/загружать четвёрки для собственных экспериментов.

In [ ]:
# Способ A: оставьте False и выберите ZIP в диалоге загрузки Colab.
# Способ B: поставьте True и укажите путь к ZIP на Google Drive.
USE_GOOGLE_DRIVE = False
DRIVE_ZIP_PATH = '/content/drive/MyDrive/ai_phonetics_data_v1.zip'

from pathlib import Path
import zipfile

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    zip_path = Path(DRIVE_ZIP_PATH)
else:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Выберите ровно один ZIP-файл датасета')
    zip_path = Path('/content') / next(iter(uploaded))

if not zip_path.is_file():
    raise FileNotFoundError(zip_path)
EXTRACT_DIR = Path('/content/ai_phonetics_data')
EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(EXTRACT_DIR)

def find_root(path):
    matches = list(path.rglob('dataset_metadata.json'))
    if len(matches) != 1: raise FileNotFoundError('dataset_metadata.json не найден')
    return matches[0].parent

DATA_ROOT = find_root(EXTRACT_DIR)
BSDS_ROOT = DATA_ROOT / 'BSDS500' / 'data'
print('DATA_ROOT:', DATA_ROOT)
print('Размер распакованного набора: %.1f MiB' % (sum(p.stat().st_size for p in DATA_ROOT.rglob('*') if p.is_file()) / 1024**2))


In [ ]:
import json
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image, ImageDraw, ImageFont
from torch.utils.data import DataLoader, Dataset

def normalise(x):
    maximum = x.max()
    if maximum <= 1e-8: raise ValueError('All-black target image')
    return x / maximum

def as_tensor(image):
    return normalise(torch.from_numpy(np.asarray(image.convert('L'), dtype=np.float32) / 255.0))

def draw_glyph(canvas, rng, value):
    char = '0123456789ABCDEFGHJKLMNPQRSTUVWXYZ'[int(rng.integers(0, 34))]; size = int(rng.integers(22, 66))
    try: font = ImageFont.truetype('DejaVuSans.ttf', size=size)
    except OSError: font = ImageFont.load_default()
    glyph = Image.new('L', (size * 2, size * 2), 0); d = ImageDraw.Draw(glyph); box = d.textbbox((0, 0), char, font=font)
    d.text((-box[0], -box[1]), char, fill=value, font=font); box = glyph.getbbox()
    if box is not None:
        glyph = glyph.crop(box).rotate(float(rng.uniform(-25, 25)), resample=Image.Resampling.BILINEAR, expand=True)
        canvas.paste(glyph, (int(rng.integers(0, max(1, 129 - glyph.width))), int(rng.integers(0, max(1, 129 - glyph.height)))), glyph)

def synthetic(seed):
    rng = np.random.default_rng(int(seed)); canvas = Image.new('L', (128, 128), 0); d = ImageDraw.Draw(canvas)
    for _ in range(int(rng.integers(1, 5))):
        kind = str(rng.choice(['ellipse', 'rectangle', 'line', 'ring', 'point', 'glyph'])); value, width = int(rng.integers(80, 256)), int(rng.integers(1, 7))
        x0, x1 = sorted(rng.integers(4, 124, size=2).tolist()); y0, y1 = sorted(rng.integers(4, 124, size=2).tolist()); x1, y1 = min(127, max(x1, x0 + 8)), min(127, max(y1, y0 + 8))
        if kind == 'ellipse': d.ellipse((x0, y0, x1, y1), fill=value if rng.random() < .5 else None, outline=value, width=width)
        elif kind == 'rectangle': d.rectangle((x0, y0, x1, y1), fill=value if rng.random() < .5 else None, outline=value, width=width)
        elif kind == 'line': d.line((x0, y0, x1, y1), fill=value, width=width)
        elif kind == 'ring': d.ellipse((x0, y0, x1, y1), outline=value, width=width)
        elif kind == 'point':
            r = int(rng.integers(2, 10)); cx, cy = int(rng.integers(r, 128-r)), int(rng.integers(r, 128-r)); d.ellipse((cx-r, cy-r, cx+r, cy+r), fill=value)
        else: draw_glyph(canvas, rng, value)
    return as_tensor(canvas)

class QuadDataset(Dataset):
    def __init__(self, split):
        manifest = DATA_ROOT / 'manifests' / f'{split}.jsonl'
        self.records = [json.loads(line) for line in manifest.read_text(encoding='utf-8').splitlines() if line]
    def __len__(self): return len(self.records)
    def get_with_sources(self, index):
        record = self.records[index]; images = []
        for item in record['items']:
            if item['source'] == 'synthetic': images.append(synthetic(item['seed']))
            else:
                with Image.open(BSDS_ROOT / item['path']) as image:
                    x, y = item['crop']['x'], item['crop']['y']; images.append(as_tensor(image.crop((x, y, x + 128, y + 128))))
        return torch.stack(images).float(), record['items']
    def __getitem__(self, index): return self.get_with_sources(index)[0]

train_set, val_set, test_set = QuadDataset('train'), QuadDataset('val'), QuadDataset('test_internal')
print('split sizes:', len(train_set), len(val_set), len(test_set))


In [ ]:
def show_quad(index=None, split='train'):
    dataset = {'train': train_set, 'val': val_set, 'test_internal': test_set}[split]
    index = random.randrange(len(dataset)) if index is None else index
    quad, specs = dataset.get_with_sources(index)
    fig, axes = plt.subplots(1, 4, figsize=(12, 3))
    for channel, axis in enumerate(axes):
        axis.imshow(quad[channel], cmap='gray', vmin=0, vmax=1)
        axis.set_title(f"{channel + 1}: {specs[channel]['source']}")
        axis.axis('off')
    fig.suptitle(f'{split}, quad #{index}; shape={tuple(quad.shape)}')
    plt.show()
    return quad, specs

quad, specs = show_quad()
print(specs)

# Меняйте index/split, чтобы изучать конкретные примеры:
# quad, specs = show_quad(index=17, split='val')


In [ ]:
# Готовый batch для ваших экспериментов или собственной модели.
loader = DataLoader(train_set, batch_size=16, shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available())
batch = next(iter(loader))
print('batch:', tuple(batch.shape), batch.dtype, 'range:', float(batch.min()), float(batch.max()))

# Пример: корреляция между первой и второй целями в каждом quad.
def pearson(a, b):
    a = a.flatten(1); b = b.flatten(1); a = a - a.mean(1, keepdim=True); b = b - b.mean(1, keepdim=True)
    return (a * b).sum(1) / (a.square().sum(1).sqrt() * b.square().sum(1).sqrt() + 1e-8)

print('CC(image 1, image 2):', pearson(batch[:, 0], batch[:, 1]).mean().item())
